# Pretrained and neural models on the 2025-26 RTM window (Kaggle GPU)

Four models on the same 150 days, folds and scores as the CPU model comparison, one cell each:
Chronos-Bolt and TimesFM (pretrained, no training on this data) and N-HiTS and TFT (trained here).

* **Chronos-Bolt, TimesFM:** for each forecast day D the model reads the filled RTM price up to the
  end of day D - 1 and forecasts the 96 blocks of D. They see no other input.
* **N-HiTS, TFT:** trained once per fold on every earlier day, then used for the 30 days of the
  fold without retraining. Inputs: the RTM price history, and for day D the DAM price, the block
  and the weekday, which are all known when day D is forecast.

**Intraday task.** A second task is added after the day-ahead scores: the same four models,
forecasting a block 1, 2, 4 or 8 blocks after the latest known RTM block (see the section
"Intraday task").

**Saving.** Each model cell writes a checkpoint after every fold to `/kaggle/working/checkpoints`
and its full predictions to `/kaggle/working/predictions_<model>.parquet` the moment it finishes.
Running a cell again skips the folds that already have a checkpoint, so a failure loses at most the
fold that was running. Download the finished files as you go. To keep them after the session ends,
use Save Version.

**Input:** a private Kaggle dataset holding `rtm_dam_prices.parquet`, written by
`scripts/export_kaggle_inputs.py`. It has RTM and DAM prices and fold labels only.

**Status:** written without a GPU. The model calls follow the public descriptions of the libraries
and have not been run yet. The first run decides whether they need changing.

In [ ]:
%pip install -q chronos-forecasting "timesfm[torch]"

In [ ]:
import shutil
import time
import traceback
from pathlib import Path

import numpy as np
import pandas as pd

CONTEXT = 2016  # blocks of history per forecast for the pretrained models: 21 days
HORIZON = 96
BATCH = 16
CAP = 10000.0
CAP_SHARE = 0.95
OUT = Path("/kaggle/working")
CKPT = OUT / "checkpoints"
CKPT.mkdir(parents=True, exist_ok=True)
RUN = ["chronos_bolt", "timesfm", "nhits", "tft"]
CHRONOS_ID = "amazon/chronos-bolt-base"
TIMESFM_ID = "google/timesfm-2.5-200m-pytorch"

In [ ]:
found = sorted(Path("/kaggle/input").rglob("rtm_dam_prices.parquet")) or [
    Path("rtm_dam_prices.parquet")
]
data = pd.read_parquet(found[0]).sort_values(["date", "block"]).reset_index(drop=True)
history = data["rtm_price_filled"].to_numpy(dtype=float)
first_index = data.groupby("date").head(1)
first_row = {day: int(i) for i, day in zip(first_index.index, first_index["date"], strict=True)}
fold_days = {
    int(k): sorted(data.loc[data["fold"] == k, "date"].unique())
    for k in sorted(data["fold"].dropna().unique())
}
print(found[0], len(data), "rows, folds:", {k: len(v) for k, v in fold_days.items()})

In [ ]:
def labelled(day, row):
    """One day of predictions in the format of the model comparison."""
    rows = data[data["date"] == day]
    part = rows[["date", "block", "rtm_price", "fold", "period"]].rename(
        columns={"rtm_price": "actual"}
    )
    part["prediction"] = row[part["block"].to_numpy() - 1]
    return part


def checkpointed(name, fold, compute):
    """The fold's predictions from its checkpoint, or computed and written to it at once."""
    path = CKPT / f"{name}_fold{fold}.parquet"
    if path.exists():
        print(f"{name} fold {fold}: checkpoint found, skipped")
        return pd.read_parquet(path)
    started = time.time()
    part = compute()
    part.to_parquet(path, index=False)
    print(f"{name} fold {fold}: {time.time() - started:.0f}s, saved")
    return part


def finish(name, parts):
    """Write the model's full predictions as soon as its last fold is done."""
    frame = pd.concat(parts, ignore_index=True)
    frame["model"] = name
    frame["fold"] = frame["fold"].astype(int)
    frame.to_parquet(OUT / f"predictions_{name}.parquet", index=False)
    print("wrote", OUT / f"predictions_{name}.parquet", len(frame), "rows")
    return frame


def contexts_for(batch):
    return [history[max(0, first_row[day] - CONTEXT) : first_row[day]] for day in batch]


def forecast_days(predict, name):
    """Forecast every fold day with a model that reads a context. ``predict`` takes a list of
    arrays and returns (days, 96)."""
    parts = []
    for fold, days in fold_days.items():

        def compute(days=days):
            pieces = []
            for first in range(0, len(days), BATCH):
                batch = days[first : first + BATCH]
                values = np.asarray(predict(contexts_for(batch)), dtype=float)
                if values.shape != (len(batch), HORIZON):
                    raise ValueError(
                        f"{name}: got {values.shape}, expected {(len(batch), HORIZON)}"
                    )
                pieces += [labelled(day, row) for day, row in zip(batch, values, strict=True)]
            return pd.concat(pieces, ignore_index=True)

        parts.append(checkpointed(name, fold, compute))
    return finish(name, parts)


def yesterday(contexts):
    return np.stack([c[-HORIZON:] for c in contexts])

## Check: yesterday's price, built the same way as the comparison's own

Its scores must match the `naive_yesterday` row of the comparison.

In [ ]:
forecast_days(yesterday, "naive_yesterday_check")

## Chronos-Bolt (pretrained)

In [ ]:
if "chronos_bolt" in RUN:
    try:
        import torch
        from chronos import BaseChronosPipeline

        pipeline = BaseChronosPipeline.from_pretrained(
            CHRONOS_ID,
            device_map="cuda" if torch.cuda.is_available() else "cpu",
            torch_dtype=torch.bfloat16,
        )

        def chronos_predict(contexts):
            tensors = [torch.tensor(c, dtype=torch.float32) for c in contexts]
            forecast = np.asarray(pipeline.predict(tensors, prediction_length=HORIZON).float())
            return forecast[:, forecast.shape[1] // 2, :]  # the median quantile

        forecast_days(chronos_predict, "chronos_bolt")
    except Exception:
        traceback.print_exc()

## TimesFM (pretrained)

In [ ]:
if "timesfm" in RUN:
    try:
        import timesfm

        model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(TIMESFM_ID)
        model.compile(
            timesfm.ForecastConfig(
                max_context=CONTEXT,
                max_horizon=HORIZON,
                normalize_inputs=True,
                use_continuous_quantile_head=True,
            )
        )

        def timesfm_predict(contexts):
            point, _ = model.forecast(horizon=HORIZON, inputs=contexts)
            return np.asarray(point)

        forecast_days(timesfm_predict, "timesfm")
    except Exception:
        traceback.print_exc()

## Neural models: shared setup

Installs the library in its own cell, because it may change the versions of packages the two
pretrained models already loaded. Run the cells above first.

In [ ]:
%pip install -q neuralforecast

In [ ]:
frame = data.copy()
frame["unique_id"] = "rtm"
frame["ds"] = frame["timestamp"].dt.tz_localize(None)  # local clock time, no time zone
frame["y"] = frame["rtm_price_filled"]
frame["weekday"] = frame["date"].dt.day_of_week.astype(float)
frame["block"] = frame["block"].astype(float)
EXOG = ["dam_price", "block", "weekday"]
long = frame[["unique_id", "ds", *EXOG, "y"]]
lookup = frame[["ds", "date", "rtm_price", "fold", "period"]].rename(
    columns={"rtm_price": "actual"}
)


def neural_forecasts(name, make_model):
    """Train once per fold on the days before it and forecast each of its days.

    The data given to the library ends with the last day of the fold, so its last windows of 96
    blocks are the fold's days. The model is not retrained inside the fold.
    """
    from neuralforecast import NeuralForecast

    parts = []
    for fold, days in fold_days.items():

        def compute(days=days):
            end = frame.loc[frame["date"] == days[-1], "ds"].max()
            nf = NeuralForecast(models=[make_model()], freq="15min")
            cv = nf.cross_validation(
                df=long[long["ds"] <= end], n_windows=len(days), step_size=HORIZON, refit=False
            )
            column = [c for c in cv.columns if c not in ("unique_id", "ds", "cutoff", "y")][0]
            part = cv[["ds", column]].rename(columns={column: "prediction"}).merge(lookup, on="ds")
            part["block"] = (part["ds"].dt.hour * 4 + part["ds"].dt.minute // 15 + 1).astype(int)
            return part[["date", "block", "actual", "fold", "period", "prediction"]]

        parts.append(checkpointed(name, fold, compute))
    return finish(name, parts)

## N-HiTS (trained here)

In [ ]:
if "nhits" in RUN:
    try:
        from neuralforecast.losses.pytorch import MAE
        from neuralforecast.models import NHITS

        def make_nhits():
            return NHITS(
                h=HORIZON,
                input_size=HORIZON * 7,
                futr_exog_list=EXOG,
                loss=MAE(),
                scaler_type="standard",
                max_steps=1500,
                random_seed=0,
                devices=1,  # one GPU: the default on two GPUs forks and fails
            )

        neural_forecasts("nhits", make_nhits)
    except Exception:
        traceback.print_exc()

## TFT (trained here)

In [ ]:
if "tft" in RUN:
    try:
        import gc

        import torch
        from neuralforecast.losses.pytorch import MAE
        from neuralforecast.models import TFT

        gc.collect()
        torch.cuda.empty_cache()  # free what an earlier failed run left

        def make_tft():
            return TFT(
                h=HORIZON,
                input_size=HORIZON * 7,
                futr_exog_list=EXOG,
                loss=MAE(),
                hidden_size=64,
                batch_size=32,
                windows_batch_size=64,  # the default 1024 needs about 9 GB for attention
                inference_windows_batch_size=64,
                scaler_type="robust",
                max_steps=1000,
                random_seed=0,
                devices=1,  # one GPU: the default on two GPUs forks and fails
            )

        neural_forecasts("tft", make_tft)
    except Exception:
        traceback.print_exc()

## Scores

In [ ]:
def scores(frame):
    rows = {}
    for name, part in frame[frame["actual"].notna()].groupby("model"):
        a, p = part["actual"].to_numpy(), part["prediction"].to_numpy()
        at_cap, called = a >= CAP, p >= CAP_SHARE * CAP
        hits = int((at_cap & called).sum())
        prec = hits / max(int(called.sum()), 1)
        rec = hits / max(int(at_cap.sum()), 1)
        rows[name] = {
            "MAE": np.mean(np.abs(a - p)),
            "RMSE": np.sqrt(np.mean((a - p) ** 2)),
            "sMAPE": np.mean(2 * np.abs(a - p) / (np.abs(a) + np.abs(p))) * 100,
            "cap_f1": 2 * prec * rec / (prec + rec) if prec + rec > 0 else float("nan"),
        }
    return pd.DataFrame(rows).T


files = sorted(OUT.glob("predictions_*.parquet"))
everything = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
print("models:", sorted(everything["model"].unique()))
for label, part in [
    ("test", everything[everything["period"] == "test"]),
    ("validation", everything[everything["period"] == "validation"]),
    *[(f"fold {k}", everything[everything["fold"] == k]) for k in fold_days],
]:
    print(f"== {label} ==")
    print(scores(part).round(2).to_string())

## Intraday task

The same four models on a second task. Day D is still forecast block by block, but now the RTM
prices up to `lead` blocks before the target block are known (`lead` is 1, 2, 4 and 8 blocks, so up
to two hours). The latest known block may lie on the day before. Which lead is realistic depends on
the exchange's gate-closure and result times, which are not confirmed, so all four are scored.

* **Chronos-Bolt, TimesFM:** for every block, the model reads the filled RTM price up to the latest
  known block and forecasts the next 8 blocks. One forecast per origin block serves all four leads.
* **N-HiTS, TFT:** trained once per fold on the rows up to 8 blocks before the fold, with a horizon
  of 8 blocks, then used for every origin in the fold without retraining. Inputs as before: the RTM
  price history and, for the forecast blocks, the DAM price, the block and the weekday.

Files are `intraday_<model>.parquet` (one file, a `lead` column) so they do not mix with the
day-ahead `predictions_<model>.parquet` files. Checkpoints after every fold work as before.
Download these files into `.project-local/results_intraday/` (see `kaggle/README.md`).

In [ ]:
LEADS = [1, 2, 4, 8]
MAXLEAD = max(LEADS)
BATCH_ORIGINS = 64
RUN_INTRADAY = ["chronos_bolt", "timesfm", "nhits", "tft"]


def fold_rows(fold):
    """Row positions in ``data`` of the days of one fold."""
    return np.flatnonzero(data["fold"].eq(fold).fillna(False).to_numpy(dtype=bool))


def intraday_part(rows, lead, values):
    part = data.loc[rows, ["date", "block", "rtm_price", "fold", "period"]].rename(
        columns={"rtm_price": "actual"}
    )
    part["prediction"] = values
    part["lead"] = lead
    return part


def finish_intraday(name, parts):
    """Write the model's intraday predictions as soon as its last fold is done."""
    frame = pd.concat(parts, ignore_index=True)
    frame["model"] = name
    frame["fold"] = frame["fold"].astype(int)
    path = OUT / f"intraday_{name}.parquet"
    frame.to_parquet(path, index=False)
    print("wrote", path, len(frame), "rows")
    return frame


def intraday_pretrained(predict, name):
    """For each fold, forecast the next ``MAXLEAD`` blocks from every origin block that a target
    block of the fold needs, and take step ``lead`` of the origin ``lead`` blocks before the target.
    ``predict`` takes a list of arrays and returns (origins, MAXLEAD)."""
    parts = []
    for fold in fold_days:

        def compute(fold=fold):
            rows = fold_rows(fold)
            origins = np.unique(np.concatenate([rows - lead for lead in LEADS]))
            steps = {}
            for first in range(0, len(origins), BATCH_ORIGINS):
                chunk = origins[first : first + BATCH_ORIGINS]
                contexts = [history[max(0, o + 1 - CONTEXT) : o + 1] for o in chunk]
                values = np.asarray(predict(contexts), dtype=float)
                if values.shape != (len(chunk), MAXLEAD):
                    want = (len(chunk), MAXLEAD)
                    raise ValueError(f"{name}: got {values.shape}, expected {want}")
                steps.update(zip(chunk.tolist(), values, strict=True))
            pieces = [
                intraday_part(rows, lead, [steps[int(t) - lead][lead - 1] for t in rows])
                for lead in LEADS
            ]
            return pd.concat(pieces, ignore_index=True)

        parts.append(checkpointed(f"{name}_intraday", fold, compute))
    return finish_intraday(name, parts)


def last_value(contexts):
    return np.stack([np.repeat(c[-1], MAXLEAD) for c in contexts])

### Check: the last known price, built the same way as the comparison's own

Its scores must match the `naive_last` row of the intraday comparison.

In [ ]:
intraday_pretrained(last_value, "naive_last_check")

### Intraday: Chronos-Bolt

In [ ]:
if "chronos_bolt" in RUN_INTRADAY:
    try:
        import torch
        from chronos import BaseChronosPipeline

        pipe = globals().get("pipeline") or BaseChronosPipeline.from_pretrained(
            CHRONOS_ID,
            device_map="cuda" if torch.cuda.is_available() else "cpu",
            torch_dtype=torch.bfloat16,
        )

        def chronos_short(contexts):
            tensors = [torch.tensor(c, dtype=torch.float32) for c in contexts]
            forecast = np.asarray(pipe.predict(tensors, prediction_length=MAXLEAD).float())
            return forecast[:, forecast.shape[1] // 2, :]  # the median quantile

        intraday_pretrained(chronos_short, "chronos_bolt")
    except Exception:
        traceback.print_exc()

### Intraday: TimesFM

In [ ]:
if "timesfm" in RUN_INTRADAY:
    try:
        import timesfm

        tfm = globals().get("model")
        if tfm is None:
            tfm = timesfm.TimesFM_2p5_200M_torch.from_pretrained(TIMESFM_ID)
            tfm.compile(
                timesfm.ForecastConfig(
                    max_context=CONTEXT,
                    max_horizon=HORIZON,
                    normalize_inputs=True,
                    use_continuous_quantile_head=True,
                )
            )

        def timesfm_short(contexts):
            point, _ = tfm.forecast(horizon=MAXLEAD, inputs=contexts)
            return np.asarray(point)

        intraday_pretrained(timesfm_short, "timesfm")
    except Exception:
        traceback.print_exc()

### Intraday: neural models

`neural_intraday` trains once per fold on the rows up to 8 blocks before the fold. Every window
that starts between 8 blocks before the fold and its last block is forecast without retraining, so
no target block is ever inside the training rows. For the last fold the data ends with the fold,
so its final blocks at leads below 8 have no window; scoring uses only the blocks that every model
has.

In [ ]:
def neural_intraday(name, make_model):
    from neuralforecast import NeuralForecast

    parts = []
    for fold in fold_days:

        def compute(fold=fold):
            rows = fold_rows(fold)
            first, last = int(rows[0]), int(rows[-1])
            end = min(last + MAXLEAD - 1, len(long) - 1)  # last row the windows may read
            windows = end - first + 1  # cutoffs from first - MAXLEAD to end - MAXLEAD
            nf = NeuralForecast(models=[make_model()], freq="15min")
            cv = nf.cross_validation(
                df=long.iloc[: end + 1], n_windows=windows, step_size=1, refit=False
            )
            column = [c for c in cv.columns if c not in ("unique_id", "ds", "cutoff", "y")][0]
            cv["lead"] = ((cv["ds"] - cv["cutoff"]) / pd.Timedelta(minutes=15)).round().astype(int)
            cv = cv[cv["lead"].isin(LEADS)]
            part = cv[["ds", "lead", column]].rename(columns={column: "prediction"})
            part = part.merge(lookup, on="ds")
            part = part[part["fold"].eq(fold).fillna(False).astype(bool)]
            part["block"] = (part["ds"].dt.hour * 4 + part["ds"].dt.minute // 15 + 1).astype(int)
            return part[["date", "block", "actual", "fold", "period", "prediction", "lead"]]

        parts.append(checkpointed(f"{name}_intraday", fold, compute))
    return finish_intraday(name, parts)

### Intraday: N-HiTS

In [ ]:
if "nhits" in RUN_INTRADAY:
    try:
        from neuralforecast.losses.pytorch import MAE
        from neuralforecast.models import NHITS

        def make_nhits_intraday():
            return NHITS(
                h=MAXLEAD,
                input_size=HORIZON * 7,
                futr_exog_list=EXOG,
                loss=MAE(),
                scaler_type="standard",
                max_steps=1500,
                random_seed=0,
                devices=1,  # one GPU: the default on two GPUs forks and fails
            )

        neural_intraday("nhits", make_nhits_intraday)
    except Exception:
        traceback.print_exc()

### Intraday: TFT

In [ ]:
if "tft" in RUN_INTRADAY:
    try:
        import gc

        import torch
        from neuralforecast.losses.pytorch import MAE
        from neuralforecast.models import TFT

        gc.collect()
        torch.cuda.empty_cache()  # free what an earlier failed run left

        def make_tft_intraday():
            return TFT(
                h=MAXLEAD,
                input_size=HORIZON * 7,
                futr_exog_list=EXOG,
                loss=MAE(),
                hidden_size=64,
                batch_size=32,
                windows_batch_size=64,  # the default 1024 needs about 9 GB for attention
                inference_windows_batch_size=64,
                scaler_type="robust",
                max_steps=1000,
                random_seed=0,
                devices=1,  # one GPU: the default on two GPUs forks and fails
            )

        neural_intraday("tft", make_tft_intraday)
    except Exception:
        traceback.print_exc()

### Intraday scores

In [ ]:
files = sorted(OUT.glob("intraday_*.parquet"))
if files:
    pooled = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
    print("models:", sorted(pooled["model"].unique()))
    for lead in LEADS:
        for period in ("test", "validation"):
            part = pooled[(pooled["lead"] == lead) & (pooled["period"] == period)]
            if part.empty:
                continue
            # only the blocks that every model has, so the models are scored on the same rows
            seen = part.groupby(["date", "block"])["model"].transform("nunique")
            part = part[seen == part["model"].nunique()]
            table = scores(part)
            if "naive_last_check" in table.index:
                table["rMAE"] = table["MAE"] / table.loc["naive_last_check", "MAE"]
            print(f"== lead {lead}, {period}, {len(part) // part['model'].nunique()} blocks ==")
            print(table.round(2).to_string())

In [ ]:
# Bundle only the finished prediction files. The library also writes model weights and logs
# under /kaggle/working (several GB after N-HiTS and TFT); they are not needed, so they are
# deleted first and the disk does not fill up. The small checkpoints folder is kept.
import zipfile

for junk in ("lightning_logs", "checkpoints_nf"):
    shutil.rmtree(OUT / junk, ignore_errors=True)
for weights in OUT.rglob("*.ckpt"):
    weights.unlink()

bundle = OUT / "predictions_bundle.zip"
bundle.unlink(missing_ok=True)
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(OUT.glob("*.parquet")):
        zf.write(path, path.name)
print("bundle:", bundle, round(bundle.stat().st_size / 1e6, 1), "MB")